# Multi-Phase Comparative Analysis & Pareto Verification

This interactive notebook provides comprehensive post-processing, comparative analytics, and verification for the **200 MeV Electron Injector Linac** optimization across:
- **Phase 1**: Scalarized Bayesian Optimization (`qLogNEI` / Weighted Merit Function)
- **Phase 2**: Unconstrained Multi-Objective BO (`qLogNEHVI` / Hypervolume)
- **Phase 3**: Constraint-Aware Multi-Objective BO (`qLogNEHVI` + Tensor Outcome Constraints)

It loads evaluation histories, computes unified hypervolume metrics under a shared reference point, plots 2D/3D Pareto frontier comparisons, evaluates constraint feasibility margins, and displays independent rerun audit results.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is in python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from mobo_linac.config import load_config
from mobo_linac.io.results import load_results, results_to_dataframe
from mobo_linac.metrics.hypervolume import HypervolumeTracker, compute_reference_point
from mobo_linac.plotting import (
    plot_hypervolume_comparison,
    plot_pareto_front_comparison,
    plot_feasibility_rate,
    plot_constraint_violins,
    plot_objective_evolution,
)
from scripts.run_comparison_and_verification import load_phase_results, generate_three_phase_report, verify_pareto_candidates

config = load_config(project_root / "configs" / "mobo_200MeV.yaml")
print(f"Loaded Linac Configuration: {config.name} (v{config.version})")

## 1. Specify Result Directories

Point to completed simulation output directories (either from `results/full_production` CLI runs or `results_notebooks/full_production` notebook runs):

In [ ]:
# Set base directories (automatically checks results/full_production then results_notebooks/full_production)
base_production = project_root / "results" / "full_production"
base_notebooks = project_root / "results_notebooks" / "full_production"
base_dir = base_production if base_production.exists() else base_notebooks

p1_dir = base_dir / "phase1_scalarized"
p2_dir = base_dir / "phase2_unconstrained"
p3_dir = base_dir / "phase3_constrained"
output_analysis_dir = base_dir / "analysis"
output_analysis_dir.mkdir(parents=True, exist_ok=True)

print(f"Analyzing campaign results from: {base_dir}")
print(f"  - Phase 1: {p1_dir} (exists: {p1_dir.exists()})")
print(f"  - Phase 2: {p2_dir} (exists: {p2_dir.exists()})")
print(f"  - Phase 3: {p3_dir} (exists: {p3_dir.exists()})")

## 2. Load Evaluation Records & DataFrames

In [ ]:
res_p1 = load_phase_results(p1_dir, config) if p1_dir.exists() else []
res_p2 = load_phase_results(p2_dir, config) if p2_dir.exists() else []
res_p3 = load_phase_results(p3_dir, config) if p3_dir.exists() else []

df_p1 = results_to_dataframe(res_p1) if res_p1 else pd.DataFrame()
df_p2 = results_to_dataframe(res_p2) if res_p2 else pd.DataFrame()
df_p3 = results_to_dataframe(res_p3) if res_p3 else pd.DataFrame()

print(f"Loaded evaluation counts:")
print(f"  - Phase 1 (Scalarized):    {len(df_p1)} samples")
print(f"  - Phase 2 (Unconstrained): {len(df_p2)} samples")
print(f"  - Phase 3 (Constrained):   {len(df_p3)} samples")

## 3. Hypervolume Progression Comparison

Compare hypervolume growth across iterations under a shared, unified reference point in model space:

In [ ]:
if res_p2 and res_p3:
    # Trackers
    from mobo_linac.metrics.hypervolume import compute_reference_point, HypervolumeTracker
    all_res = res_p1 + res_p2 + res_p3
    all_df = results_to_dataframe(all_res)
    
    # Compute shared reference point
    ref_point = torch.tensor([-4.0e-6, -4.0e-6, -1.2e6], dtype=torch.double)
    
    t_p2 = HypervolumeTracker(ref_point=ref_point)
    for r in res_p2:
        t_p2.update(r)
        
    t_p3 = HypervolumeTracker(ref_point=ref_point)
    for r in res_p3:
        t_p3.update(r)
        
    fig = plot_hypervolume_comparison(
        trackers={"Phase 2 (Unconstrained)": t_p2, "Phase 3 (Constrained)": t_p3},
        save_path=output_analysis_dir / "hypervolume_comparison.png",
    )
    plt.show()
else:
    print("Run data not yet available for Phase 2 and Phase 3.")

## 4. Pareto Frontier 2D & 3D Comparisons

Overlay trade-off frontiers between horizontal emittance $\varepsilon_{n,x}$, vertical emittance $\varepsilon_{n,y}$, and energy spread $\sigma_E$:

In [ ]:
results_dict = {}
if res_p1:
    results_dict["Phase 1: Scalarized"] = res_p1
if res_p2:
    results_dict["Phase 2: Unconstrained"] = res_p2
if res_p3:
    results_dict["Phase 3: Constrained"] = res_p3

if results_dict:
    fig = plot_pareto_front_comparison(
        results_dict=results_dict,
        save_path=output_analysis_dir / "pareto_comparison_2d.png",
    )
    plt.show()
else:
    print("No results to plot.")

## 5. Constraint Satisfaction & Feasibility Analysis

Evaluate constraint feasibility rates and distribution of transmission and beam constraints ($Q \ge 0.95\,\text{nC}$, $E \ge 195\,\text{MeV}$):

In [ ]:
if res_p3:
    fig_feas = plot_feasibility_rate(
        results=res_p3,
        save_path=output_analysis_dir / "phase3_feasibility_rate.png",
    )
    plt.show()
    
    fig_viol = plot_constraint_violins(
        results=res_p3,
        constraints_config=config.constraints,
        save_path=output_analysis_dir / "phase3_constraint_violins.png",
    )
    plt.show()
else:
    print("Phase 3 results required for constraint analysis.")

## 6. Independent Pareto Rerun Verification Audit

Re-evaluate Pareto non-dominated candidates in isolated working directories to check for surrogate consistency and simulation repeatability:

In [ ]:
if res_p3:
    verif_records = verify_pareto_candidates(res_p3, config, output_analysis_dir)
    print(f"Verified {len(verif_records)} Pareto optimal candidates.")
    
    # Generate full comparative analysis markdown report
    report_path = generate_three_phase_report(
        p1_dir=p1_dir, res_p1=res_p1,
        p2_dir=p2_dir, res_p2=res_p2,
        p3_dir=p3_dir, res_p3=res_p3,
        verification_records=verif_records,
        output_dir=output_analysis_dir,
    )
    print(f"Comprehensive Comparative Report written to: {report_path}")
else:
    print("Phase 3 results required for verification audit.")